In [1]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# CNN Encoder
# -------------------------
class CNNEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # 128 → 64
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # 64 → 32
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # 32 → 16
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), # 16 → 8
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)) #nn.AdaptiveAvgPool2d((2,2)) if model struggles
        )

        self.fc = nn.Linear(128, feature_dim) 

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1) # flatten
        feat = self.fc(x)
        return feat


# -------------------------
# CNN + LSTM Model
# -------------------------
class CNN_LSTM(nn.Module):
    def __init__(self, feature_dim=128, hidden_dim=128):
        super().__init__()

        self.encoder = CNNEncoder(feature_dim)

        self.lstm = nn.LSTM(feature_dim, hidden_dim, batch_first=True)

        self.fc_out = nn.Linear(hidden_dim, 4)

    def forward_step(self, img, hidden):
        # img: (1, 3, H, W)

        feat = self.encoder(img)          # (1, feature_dim)
        feat = feat.unsqueeze(1)          # (1, 1, feature_dim)

        out, hidden = self.lstm(feat, hidden)

        pred = self.fc_out(out)           # (1, 1, 4)

        return pred, hidden

    def init_hidden(self):
        h = torch.zeros(1, 1, 128).to(DEVICE)
        c = torch.zeros(1, 1, 128).to(DEVICE)
        return (h, c)
    
import matplotlib.pyplot as plt
import numpy as np

def render_frame_tensor(traj_point, target):
    lx, ly, rx, ry = traj_point

    fig = plt.figure(figsize=(2,2), dpi=64)

    plt.scatter(lx, ly, c='blue', s=50)
    plt.scatter(rx, ry, c='green', s=50)
    plt.scatter(target[0], target[1], c='red', marker='^', s=80)

    plt.xlim(-0.8, 0.8)
    plt.ylim(-0.1, 0.8)

    plt.axis('off')

    fig.canvas.draw()

    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))

    plt.close()

    # to tensor
    img = torch.from_numpy(img).float() / 255.0   # normalize here
    img = img.permute(2, 0, 1)  # (C,H,W)

    return img

def teacher_forcing_prob(epoch, total_epochs):
    start = 1.0
    end = 0.1
    frac = min(1.0, epoch / total_epochs)
    return start + (end - start) * frac


    

In [2]:
from torch.utils.data import Dataset, DataLoader

class TrajDataset(Dataset):
    def __init__(self, images, coords):
        self.images = images
        self.coords = coords

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.coords[idx]

In [3]:
data = torch.load("trajectory_dataset.pt")

images = data["images"]
coords = data["coords"]

# split
N = len(images)
split = int(0.8 * N)

train_images = images[:split]
val_images = images[split:]

train_coords = coords[:split]
val_coords = coords[split:]

In [4]:
train_loader = DataLoader(
    TrajDataset(train_images, train_coords),
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    TrajDataset(val_images, val_coords),
    batch_size=8,
    shuffle=False
)

In [ ]:
model = CNN_LSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 50

train_losses = []
val_losses = []

for epoch in range(1, EPOCHS+1):

    model.train()
    tf_p = teacher_forcing_prob(epoch, EPOCHS)

    train_loss = 0.0

    for imgs, coords in train_loader:

        imgs = imgs.to(DEVICE)       # (B,T,C,H,W)
        coords = coords.to(DEVICE)   # (B,T,4)

        B, T = coords.shape[0], coords.shape[1]

        batch_loss = 0.0

        for b in range(B):

            hidden = model.init_hidden()

            prev_img = imgs[b,0].unsqueeze(0)
            loss_traj = 0.0

            for t in range(1, T):

                pred, hidden = model.forward_step(prev_img, hidden)

                gt = coords[b,t].view(1,1,4)

                loss = loss_fn(pred, gt)
                loss_traj += loss

                if np.random.rand() < tf_p:
                    prev_img = imgs[b,t].unsqueeze(0)
                else:
                    pred_np = pred.detach().cpu().numpy().reshape(4,)
                    prev_img = render_frame_tensor(pred_np, coords[b,-1,:2].cpu().numpy()).to(DEVICE).unsqueeze(0)

            loss_traj = loss_traj / (T-1)
            batch_loss += loss_traj

        batch_loss = batch_loss / B

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        train_loss += batch_loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for imgs, coords in val_loader:

            imgs = imgs.to(DEVICE)
            coords = coords.to(DEVICE)

            B, T = coords.shape[0], coords.shape[1]

            for b in range(B):

                hidden = model.init_hidden()
                prev_img = imgs[b,0].unsqueeze(0)

                loss_traj = 0.0

                for t in range(1, T):

                    pred, hidden = model.forward_step(prev_img, hidden)

                    gt = coords[b,t].view(1,1,4)
                    loss_traj += loss_fn(pred, gt)

                    # NO teacher forcing
                    pred_np = pred.detach().cpu().numpy().reshape(4,)
                    prev_img = render_frame_tensor(pred_np, coords[b,-1,:2].cpu().numpy()).to(DEVICE).unsqueeze(0)

                loss_traj = loss_traj / (T-1)
                val_loss += loss_traj.item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch} | Train {train_loss:.4f} | Val {val_loss:.4f} | TF {tf_p:.3f}")

C:\Users\User\AppData\Local\Temp\ipykernel_9968\3630493317.py:83: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed two minor releases later. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
C:\Users\User\AppData\Local\Temp\ipykernel_9968\3630493317.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ..\torch\csrc\utils\tensor_numpy.cpp:212.)
  img = torch.from_numpy(img).float() / 255.0   # normalize here


Epoch 1 | Train 0.0121 | Val 0.1755 | TF 0.982
Epoch 2 | Train 0.0089 | Val 0.1488 | TF 0.964
Epoch 3 | Train 0.0087 | Val 0.2033 | TF 0.946
Epoch 4 | Train 0.0084 | Val 0.1368 | TF 0.928
Epoch 5 | Train 0.0076 | Val 0.2609 | TF 0.910
Epoch 6 | Train 0.0068 | Val 0.2460 | TF 0.892


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8,5))

plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")

plt.yscale("log")   # 🔥 helps see trends clearly

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve (Log Scale)")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()